# Chapter 8: Audio & Speech Modality — From Waveforms to Multimodal LLMs

---

**Key Insight:** Once you understand the `encoder → projector → LLM` pattern from vision (Chapters 4–5), adding *any* new modality is the same architecture with a different encoder. This is why Phi-4 Multimodal and MiniCPM-o handle vision + audio + text — they just stack more encoders with more projectors feeding into the same frozen LLM.

```
Audio waveform (16kHz, T samples)
        │
        ▼  Mel spectrogram (from scratch)
   (num_mel_bins=80, time_steps=3000)
        │
        ▼  Audio Encoder (Transformer)
   (1, time_steps/2, audio_dim=512)
        │
        ▼  Projector (MLP — same as vision)
   (1, time_steps/2, llm_dim=4096)
        │
        ▼  Concatenate with text tokens → LLM
```

**Papers covered:**
- Whisper (Radford et al., 2022) — encoder-decoder for robust ASR
- Phi-4 Multimodal (Microsoft, 2025) — unified vision+audio+text
- MiniCPM-o (OpenBMB, 2025) — efficient omni-modal model

**Notebook structure:**
1. Audio basics: waveform → mel spectrogram (from scratch)
2. Audio tokenization: continuous vs discrete
3. Whisper architecture walkthrough
4. Build a mini audio transformer encoder from scratch
5. Connect audio encoder to LLM via MLP projector
6. Demo: speech understanding through the LLM
7. Unified audio-vision-text architectures
8. Bonus: speech synthesis overview (TTS)

In [ ]:
# ============================================================
# Environment setup and imports
# ============================================================
!pip install -q torch torchaudio matplotlib numpy librosa soundfile

import math
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from typing import Optional, Tuple
from dataclasses import dataclass

torch.manual_seed(42)
np.random.seed(42)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

---
# 1) Audio Basics: Waveform → Mel Spectrogram (From Scratch)

## 1.1 What Is a Waveform?

**Intuition:** Sound is a pressure wave propagating through air. A microphone converts these pressure fluctuations into a 1D discrete signal sampled at some rate (e.g., 16,000 Hz means 16,000 amplitude measurements per second).

**Sample Input → Output:**
```
Input:  A 3-second audio clip at 16kHz → tensor of shape (48000,)
Output: Visualization of the waveform as amplitude vs. time
```

**Why 16kHz?** Speech content lives primarily in 0–8kHz. By Nyquist's theorem, 16kHz sampling captures all frequencies up to 8kHz — sufficient for speech recognition. Music uses 44.1kHz or 48kHz to capture the full audible range (20Hz–20kHz).

In [ ]:
# ============================================================
# Generate a synthetic speech-like waveform for demonstration
# We combine multiple sinusoids at speech-relevant frequencies
# with amplitude modulation to mimic speech envelope patterns.
# ============================================================

SAMPLE_RATE = 16000  # 16kHz — standard for speech
DURATION = 3.0  # seconds


def generate_synthetic_speech(
    sample_rate: int = 16000,
    duration: float = 3.0,
    fundamental_freq: float = 150.0,
) -> torch.Tensor:
    """Generate a synthetic speech-like signal with harmonics and amplitude modulation.

    Real speech has a fundamental frequency (F0) plus harmonics, modulated by
    a slowly-varying amplitude envelope that encodes syllable structure.

    Args:
        sample_rate: Samples per second.
        duration: Length in seconds.
        fundamental_freq: F0 in Hz (typical male ~120Hz, female ~200Hz).

    Returns:
        Tensor of shape (num_samples,) with values in [-1, 1].
    """
    num_samples = int(sample_rate * duration)
    t = torch.linspace(0, duration, num_samples)

    # Fundamental + first 4 harmonics with decreasing amplitude
    signal = torch.zeros(num_samples)
    for harmonic_idx in range(1, 6):
        freq = fundamental_freq * harmonic_idx
        amplitude = 1.0 / harmonic_idx
        signal += amplitude * torch.sin(2 * math.pi * freq * t)

    # Amplitude modulation at ~4Hz to simulate syllable rhythm
    envelope = 0.5 * (1 + torch.sin(2 * math.pi * 4.0 * t))
    signal = signal * envelope

    # Normalize to [-1, 1]
    signal = signal / signal.abs().max()
    return signal


waveform = generate_synthetic_speech(SAMPLE_RATE, DURATION)
print(f"Waveform shape: {waveform.shape}")
print(f"Duration: {waveform.shape[0] / SAMPLE_RATE:.1f}s at {SAMPLE_RATE}Hz")

fig, axes = plt.subplots(2, 1, figsize=(12, 5))

# Full waveform
time_axis = torch.linspace(0, DURATION, waveform.shape[0])
axes[0].plot(time_axis.numpy(), waveform.numpy(), linewidth=0.3)
axes[0].set_title("Full Waveform (3 seconds)")
axes[0].set_xlabel("Time (s)")
axes[0].set_ylabel("Amplitude")

# Zoomed view — first 50ms to see individual oscillations
zoom_samples = int(0.05 * SAMPLE_RATE)
axes[1].plot(
    time_axis[:zoom_samples].numpy(), waveform[:zoom_samples].numpy(), linewidth=0.8
)
axes[1].set_title("Zoomed: First 50ms")
axes[1].set_xlabel("Time (s)")
axes[1].set_ylabel("Amplitude")

plt.tight_layout()
plt.show()

## 1.2 From Waveform to Spectrogram: The STFT

**Intuition:** A raw waveform mixes *all* frequencies together at each time step. To understand speech, we need to know *which frequencies are present at which times*. The Short-Time Fourier Transform (STFT) slides a window across the waveform and computes the DFT within each window, producing a time-frequency representation.

**Sample Input → Output:**
```
Input:  waveform of shape (num_samples,) = (48000,)
Output: complex STFT matrix of shape (num_freq_bins, num_frames)
        where num_freq_bins = n_fft//2 + 1 = 201
        and   num_frames ≈ num_samples // hop_length + 1
```

**Key parameters:**
- `n_fft`: FFT window size. Larger → better frequency resolution, worse time resolution (uncertainty principle).
- `hop_length`: How far the window slides between frames. Smaller → more frames, finer time granularity.
- `window`: Tapering function (e.g., Hann) that reduces spectral leakage at window boundaries.

**Why not just use the raw waveform?** The waveform has 16,000 samples/sec — that's 480,000 tokens for 30s of audio. A spectrogram compresses this to ~3,000 time frames while exposing the frequency structure that matters for speech.

In [ ]:
# ============================================================
# Implement STFT from scratch
# ============================================================


def hann_window(window_length: int) -> torch.Tensor:
    """Compute the Hann window function from scratch.

    The Hann window w[n] = 0.5 * (1 - cos(2π·n / (N-1))) smoothly tapers
    the signal to zero at both edges, reducing spectral leakage caused by
    the implicit rectangular truncation of the DFT.

    Args:
        window_length: Number of samples in the window.

    Returns:
        Tensor of shape (window_length,).
    """
    n = torch.arange(window_length, dtype=torch.float32)
    return 0.5 * (1.0 - torch.cos(2.0 * math.pi * n / (window_length - 1)))


def stft_from_scratch(
    waveform: torch.Tensor,
    n_fft: int = 400,
    hop_length: int = 160,
    window: Optional[torch.Tensor] = None,
) -> torch.Tensor:
    """Compute the Short-Time Fourier Transform from scratch.

    Slides a window of size n_fft across the waveform in steps of hop_length,
    applies the window function, and computes the DFT of each frame.

    Args:
        waveform: Input signal of shape (num_samples,).
        n_fft: FFT size (window length).
        hop_length: Number of samples between consecutive frames.
        window: Window function of shape (n_fft,). Defaults to Hann.

    Returns:
        Complex tensor of shape (n_fft//2 + 1, num_frames).
    """
    if window is None:
        window = hann_window(n_fft)

    # Pad the signal so the last frame is centered
    pad_length = n_fft // 2
    padded = F.pad(waveform, (pad_length, pad_length), mode="reflect")

    # Calculate number of frames
    num_frames = (padded.shape[0] - n_fft) // hop_length + 1

    # Extract overlapping frames using unfold
    # (num_samples_padded,) → (num_frames, n_fft)
    frames = padded.unfold(0, n_fft, hop_length)[:num_frames]

    # Apply window function element-wise
    # (num_frames, n_fft) * (n_fft,) → (num_frames, n_fft)
    windowed_frames = frames * window.unsqueeze(0)

    # Compute FFT along the last dimension, keep only positive frequencies
    # (num_frames, n_fft) → (num_frames, n_fft//2 + 1)
    spectrum = torch.fft.rfft(windowed_frames, n=n_fft, dim=-1)

    # Transpose to (num_freq_bins, num_frames) — conventional spectrogram layout
    return spectrum.T


# Compute STFT
N_FFT = 400
HOP_LENGTH = 160
stft_result = stft_from_scratch(waveform, n_fft=N_FFT, hop_length=HOP_LENGTH)

print(f"STFT shape: {stft_result.shape}")
print(f"  Frequency bins: {stft_result.shape[0]} (= n_fft//2 + 1 = {N_FFT // 2 + 1})")
print(f"  Time frames: {stft_result.shape[1]}")

# Compute and visualize the power spectrogram (magnitude squared)
# (num_freq_bins, num_frames)
power_spec = stft_result.abs() ** 2

plt.figure(figsize=(12, 4))
plt.imshow(
    10 * torch.log10(power_spec + 1e-10).numpy(),
    aspect="auto",
    origin="lower",
    cmap="viridis",
)
plt.colorbar(label="Power (dB)")
plt.xlabel("Time Frame")
plt.ylabel("Frequency Bin")
plt.title("Power Spectrogram (linear frequency scale)")
plt.tight_layout()
plt.show()

## 1.3 The Mel Scale and Mel Filterbank

**Intuition:** Human hearing perceives pitch *logarithmically* — the perceptual difference between 100Hz and 200Hz feels the same as between 1000Hz and 2000Hz (both are one octave). The **mel scale** maps physical frequency (Hz) to *perceived pitch* (mels), compressing high frequencies where our ears have less resolution.

$$m = 2595 \cdot \log_{10}\left(1 + \frac{f}{700}\right)$$

A **mel filterbank** is a set of overlapping triangular filters spaced uniformly on the mel scale. Applying it to a power spectrogram converts the linear frequency axis into a perceptually-motivated one.

**Sample Input → Output:**
```
Input:  Power spectrogram of shape (n_fft//2+1, num_frames) = (201, 301)
Output: Mel spectrogram of shape (num_mel_bins, num_frames) = (80, 301)
```

**Why 80 mel bins?** This is the Whisper default. It provides sufficient spectral resolution for speech without being wasteful. More bins (128) are common in music tasks.

In [ ]:
# ============================================================
# Implement mel filterbank from scratch
# ============================================================


def hz_to_mel(freq_hz: torch.Tensor) -> torch.Tensor:
    """Convert frequency in Hz to mel scale using the O'Shaughnessy formula."""
    return 2595.0 * torch.log10(1.0 + freq_hz / 700.0)


def mel_to_hz(freq_mel: torch.Tensor) -> torch.Tensor:
    """Convert mel scale back to Hz."""
    return 700.0 * (10.0 ** (freq_mel / 2595.0) - 1.0)


def create_mel_filterbank(
    num_mel_bins: int = 80,
    n_fft: int = 400,
    sample_rate: int = 16000,
    f_min: float = 0.0,
    f_max: Optional[float] = None,
) -> torch.Tensor:
    """Create a mel-scale filterbank matrix from scratch.

    Each row is a triangular filter centered at a mel-spaced frequency.
    Adjacent filters overlap, so every frequency bin contributes to exactly
    two mel bins (except at the edges), preserving energy.

    Args:
        num_mel_bins: Number of mel frequency bands.
        n_fft: FFT window size.
        sample_rate: Audio sample rate in Hz.
        f_min: Minimum frequency for the filterbank.
        f_max: Maximum frequency (defaults to sample_rate / 2).

    Returns:
        Filterbank matrix of shape (num_mel_bins, n_fft//2 + 1).
    """
    if f_max is None:
        f_max = sample_rate / 2.0

    num_freq_bins = n_fft // 2 + 1

    # Create num_mel_bins + 2 equally spaced points on the mel scale
    # (+2 for the left edge of the first filter and right edge of the last)
    mel_min = hz_to_mel(torch.tensor(f_min))
    mel_max = hz_to_mel(torch.tensor(f_max))
    mel_points = torch.linspace(mel_min.item(), mel_max.item(), num_mel_bins + 2)

    # Convert mel points back to Hz, then to FFT bin indices
    hz_points = mel_to_hz(mel_points)
    bin_indices = (hz_points / sample_rate * n_fft).long()

    # Build triangular filters
    # (num_mel_bins, num_freq_bins)
    filterbank = torch.zeros(num_mel_bins, num_freq_bins)
    for m in range(num_mel_bins):
        left = bin_indices[m]
        center = bin_indices[m + 1]
        right = bin_indices[m + 2]

        # Rising slope: left → center
        for k in range(left, center):
            if center != left:  # avoid division by zero
                filterbank[m, k] = (k - left).float() / (center - left).float()

        # Falling slope: center → right
        for k in range(center, right):
            if right != center:
                filterbank[m, k] = (right - k).float() / (right - center).float()

    return filterbank


# Create and visualize the filterbank
NUM_MEL_BINS = 80
mel_fb = create_mel_filterbank(
    num_mel_bins=NUM_MEL_BINS, n_fft=N_FFT, sample_rate=SAMPLE_RATE
)
print(f"Mel filterbank shape: {mel_fb.shape}")

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Plot a subset of filters to show triangular shape
freq_hz = torch.linspace(0, SAMPLE_RATE / 2, N_FFT // 2 + 1)
for i in range(0, NUM_MEL_BINS, 10):
    axes[0].plot(freq_hz.numpy(), mel_fb[i].numpy(), linewidth=0.8)
axes[0].set_title("Mel Filterbank (every 10th filter)")
axes[0].set_xlabel("Frequency (Hz)")
axes[0].set_ylabel("Filter weight")

# Show full filterbank as heatmap
axes[1].imshow(mel_fb.numpy(), aspect="auto", origin="lower", cmap="viridis")
axes[1].set_title("Full Mel Filterbank (80 filters)")
axes[1].set_xlabel("FFT Bin")
axes[1].set_ylabel("Mel Bin")

plt.tight_layout()
plt.show()

## 1.4 Full Mel Spectrogram Pipeline (From Scratch)

**Intuition:** Combining STFT + mel filterbank + log compression gives us the **log-mel spectrogram** — the standard input representation for modern speech models (Whisper, Wav2Vec2, etc.). The log compression is critical: raw power values span many orders of magnitude, and log compression matches human loudness perception (Weber-Fechner law).

**Complete pipeline:**
```
Waveform (num_samples,)
    → STFT → Complex spectrogram (n_fft//2+1, num_frames)
    → |·|² → Power spectrogram (n_fft//2+1, num_frames)
    → Mel filterbank → Mel spectrogram (num_mel_bins, num_frames)
    → log(·) → Log-mel spectrogram (num_mel_bins, num_frames)
```

In [ ]:
# ============================================================
# Complete mel spectrogram pipeline from scratch
# ============================================================


def compute_log_mel_spectrogram(
    waveform: torch.Tensor,
    sample_rate: int = 16000,
    n_fft: int = 400,
    hop_length: int = 160,
    num_mel_bins: int = 80,
) -> torch.Tensor:
    """Compute log-mel spectrogram from raw waveform, entirely from scratch.

    This mirrors the feature extraction in Whisper: 25ms windows (400 samples
    at 16kHz), 10ms hops (160 samples), 80 mel bins, log-scaled.

    Args:
        waveform: Raw audio signal of shape (num_samples,).
        sample_rate: Sampling rate in Hz.
        n_fft: FFT window size.
        hop_length: Hop size between consecutive frames.
        num_mel_bins: Number of mel frequency bands.

    Returns:
        Log-mel spectrogram of shape (num_mel_bins, num_frames).
    """
    # Compute STFT
    # (num_samples,) → (n_fft//2+1, num_frames)
    stft_complex = stft_from_scratch(waveform, n_fft=n_fft, hop_length=hop_length)

    # Power spectrogram
    # (n_fft//2+1, num_frames) → (n_fft//2+1, num_frames)
    power = stft_complex.abs() ** 2

    # Create mel filterbank and apply it
    # (num_mel_bins, n_fft//2+1) @ (n_fft//2+1, num_frames) → (num_mel_bins, num_frames)
    mel_fb = create_mel_filterbank(
        num_mel_bins=num_mel_bins,
        n_fft=n_fft,
        sample_rate=sample_rate,
    )
    mel_spec = mel_fb @ power

    # Log compression with small epsilon for numerical stability
    # Whisper uses log10 and clamps to a maximum dynamic range
    log_mel = torch.log10(torch.clamp(mel_spec, min=1e-10))

    # Normalize to [-1, 1] range (Whisper-style: max normalization + shift)
    log_mel = torch.maximum(log_mel, log_mel.max() - 8.0)
    log_mel = (log_mel + 4.0) / 4.0

    return log_mel


# Compute and visualize
log_mel_spec = compute_log_mel_spectrogram(waveform, SAMPLE_RATE)
print(f"Log-mel spectrogram shape: {log_mel_spec.shape}")
print(f"  Mel bins: {log_mel_spec.shape[0]}, Time frames: {log_mel_spec.shape[1]}")

fig, axes = plt.subplots(3, 1, figsize=(12, 9))

# Waveform
axes[0].plot(
    torch.linspace(0, DURATION, waveform.shape[0]).numpy(),
    waveform.numpy(),
    linewidth=0.3,
)
axes[0].set_title("Waveform")
axes[0].set_ylabel("Amplitude")

# Power spectrogram
power_db = 10 * torch.log10(stft_from_scratch(waveform).abs() ** 2 + 1e-10)
axes[1].imshow(power_db.numpy(), aspect="auto", origin="lower", cmap="viridis")
axes[1].set_title("Power Spectrogram (linear freq)")
axes[1].set_ylabel("Freq Bin")

# Log-mel spectrogram
axes[2].imshow(log_mel_spec.numpy(), aspect="auto", origin="lower", cmap="viridis")
axes[2].set_title("Log-Mel Spectrogram (80 mel bins) — Model Input")
axes[2].set_ylabel("Mel Bin")
axes[2].set_xlabel("Time Frame")

plt.tight_layout()
plt.show()

print("\n✓ Pipeline: waveform → STFT → power → mel filterbank → log → normalized")

---
# 2) Audio Tokenization: Continuous vs. Discrete

## 2.1 Two Paradigms for Representing Audio

**Intuition:** Just as vision models debate between continuous patch embeddings (ViT) and discrete visual tokens (VQVAE), audio models face the same choice:

| Approach | Representation | Used by | Pros | Cons |
|----------|---------------|---------|------|------|
| **Continuous** | Mel spectrogram frames → encoder → dense vectors | Whisper, Phi-4, MiniCPM-o | Preserves all information, differentiable end-to-end | Not directly compatible with text vocabulary |
| **Discrete** | Waveform → neural codec → token IDs | EnCodec, SoundStorm, VALL-E | Shares vocabulary with text tokens, enables AR generation | Lossy compression, codebook collapse risk |

**Key insight for multimodal LLMs:** Most speech *understanding* systems (ASR, speech-to-text) use the **continuous** path — it preserves more acoustic detail. Discrete tokens shine in speech *generation* (TTS) where you need the LLM to autoregressively produce audio.

## 2.2 Continuous: Spectrogram Frames as Tokens

In the continuous approach (used by Whisper, Phi-4 Multimodal):
```
waveform → mel spectrogram (80, T) → audio encoder → (T', D) dense vectors
```
Each time frame of the spectrogram becomes a "token" — a continuous vector of dimension D. These are projected to the LLM's embedding space via an MLP projector, identically to how ViT patch embeddings are projected.

## 2.3 Discrete: Neural Audio Codecs (EnCodec)

Neural codecs like Meta's EnCodec use a VQ-VAE architecture:
```
waveform → encoder → RVQ (residual vector quantization) → discrete codes
         → decoder → reconstructed waveform
```
**Residual Vector Quantization (RVQ)** uses multiple codebooks in sequence. The first codebook captures coarse structure; each subsequent one quantizes the *residual error* from the previous, adding fine detail.

Let's implement a simplified version of both to build intuition.

In [ ]:
# ============================================================
# Demonstrate both tokenization approaches
# ============================================================

# --- Continuous tokenization (spectrogram frames as tokens) ---


class ContinuousAudioTokenizer(nn.Module):
    """Convert mel spectrogram frames into continuous token vectors.

    Each time frame of the mel spectrogram is linearly projected to a
    token embedding. This is the simplest form of continuous audio
    tokenization — Whisper adds conv subsampling layers before this.
    """

    def __init__(self, num_mel_bins: int = 80, embed_dim: int = 512):
        super().__init__()
        # (num_mel_bins,) → (embed_dim,) per frame
        self.projection = nn.Linear(num_mel_bins, embed_dim)

    def forward(self, mel_spec: torch.Tensor) -> torch.Tensor:
        """
        Args:
            mel_spec: (batch_num, num_mel_bins, num_frames)
        Returns:
            Token embeddings: (batch_num, num_frames, embed_dim)
        """
        # Transpose so time is the sequence dimension
        # (batch_num, num_mel_bins, num_frames) → (batch_num, num_frames, num_mel_bins)
        mel_transposed = mel_spec.transpose(1, 2)

        # Project each frame to embedding space
        # (batch_num, num_frames, num_mel_bins) → (batch_num, num_frames, embed_dim)
        return self.projection(mel_transposed)


# --- Discrete tokenization (simplified VQ) ---


class VectorQuantizer(nn.Module):
    """Simplified vector quantization layer.

    Maps each continuous vector to the nearest codebook entry, producing
    discrete token IDs. Uses straight-through gradient estimator for
    backpropagation through the non-differentiable argmin.
    """

    def __init__(self, codebook_size: int = 1024, embed_dim: int = 512):
        super().__init__()
        self.codebook_size = codebook_size
        # Learnable codebook: each entry is a prototype vector
        # (codebook_size, embed_dim)
        self.codebook = nn.Embedding(codebook_size, embed_dim)
        nn.init.uniform_(self.codebook.weight, -1.0 / codebook_size, 1.0 / codebook_size)

    def forward(
        self, z: torch.Tensor
    ) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        """
        Args:
            z: Continuous embeddings (batch_num, seq_len, embed_dim)
        Returns:
            quantized: Quantized embeddings (batch_num, seq_len, embed_dim)
            token_ids: Discrete codes (batch_num, seq_len)
            commitment_loss: VQ training loss scalar
        """
        flat_z = z.reshape(-1, z.shape[-1])

        # Compute L2 distances to all codebook entries
        # (batch_num*seq_len, embed_dim) vs (codebook_size, embed_dim)
        # Using expanded form: ||z - e||² = ||z||² - 2·z·eᵀ + ||e||²
        distances = (
            flat_z.pow(2).sum(dim=-1, keepdim=True)
            - 2 * flat_z @ self.codebook.weight.T
            + self.codebook.weight.pow(2).sum(dim=-1, keepdim=True).T
        )

        # Find nearest codebook entry
        # (batch_num*seq_len, codebook_size) → (batch_num*seq_len,)
        token_ids = distances.argmin(dim=-1)

        # Look up the quantized vectors
        # (batch_num*seq_len,) → (batch_num*seq_len, embed_dim)
        quantized_flat = self.codebook(token_ids)
        quantized = quantized_flat.reshape(z.shape)
        token_ids = token_ids.reshape(z.shape[:-1])

        # Commitment loss: encourage encoder outputs to stay near codebook
        commitment_loss = F.mse_loss(z.detach(), quantized) + F.mse_loss(
            z, quantized.detach()
        )

        # Straight-through estimator: copy gradients from quantized to z
        quantized = z + (quantized - z).detach()

        return quantized, token_ids, commitment_loss


# --- Demonstrate both ---

mel_input = log_mel_spec.unsqueeze(0)  # (1, 80, num_frames)
print(f"Mel spectrogram input: {mel_input.shape}")

# Continuous tokenization
cont_tokenizer = ContinuousAudioTokenizer(num_mel_bins=80, embed_dim=512)
continuous_tokens = cont_tokenizer(mel_input)
print(f"\nContinuous tokens: {continuous_tokens.shape}")
print(f"  → Each frame is a 512-dim vector, preserving full information")

# Discrete tokenization
vq = VectorQuantizer(codebook_size=1024, embed_dim=512)
quantized, token_ids, vq_loss = vq(continuous_tokens)
print(f"\nDiscrete token IDs: {token_ids.shape}")
print(f"  → Each frame maps to one of 1024 codebook entries")
print(f"  → First 20 tokens: {token_ids[0, :20].tolist()}")
print(f"  → VQ loss: {vq_loss.item():.4f}")

## 2.4 Residual Vector Quantization (RVQ)

**Intuition:** A single codebook of 1024 entries can only represent 1024 distinct sounds — far too few. RVQ solves this by using *K* codebooks in sequence: the first quantizes the signal, the second quantizes the *residual error*, the third quantizes the residual of the residual, etc. With K=8 codebooks of size 1024 each, you get 1024⁸ ≈ 10²⁴ possible combinations.

```
z₀ = encoder(x)         # continuous embedding
q₁ = VQ₁(z₀)           # coarse quantization
r₁ = z₀ - q₁           # residual after 1st codebook
q₂ = VQ₂(r₁)           # quantize the residual
r₂ = r₁ - q₂           # residual after 2nd codebook
...                     # repeat K times
final = q₁ + q₂ + ... + qₖ  # sum all quantized components
```

In [ ]:
# ============================================================
# Residual Vector Quantization from scratch
# ============================================================


class ResidualVQ(nn.Module):
    """Residual Vector Quantization with K codebooks.

    Each codebook quantizes the residual error from all previous codebooks,
    progressively refining the representation. This is the core of EnCodec.
    """

    def __init__(
        self,
        num_quantizers: int = 4,
        codebook_size: int = 1024,
        embed_dim: int = 512,
    ):
        super().__init__()
        self.quantizers = nn.ModuleList(
            [VectorQuantizer(codebook_size, embed_dim) for _ in range(num_quantizers)]
        )

    def forward(
        self, z: torch.Tensor
    ) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        """
        Args:
            z: Continuous embeddings (batch_num, seq_len, embed_dim)
        Returns:
            quantized: Refined quantized output (batch_num, seq_len, embed_dim)
            all_ids: Token IDs from each codebook (num_quantizers, batch_num, seq_len)
            total_loss: Sum of commitment losses
        """
        residual = z
        quantized_sum = torch.zeros_like(z)
        all_ids = []
        total_loss = torch.tensor(0.0, device=z.device)

        for quantizer in self.quantizers:
            # Quantize current residual
            quantized, ids, loss = quantizer(residual)
            # Accumulate quantized output
            quantized_sum = quantized_sum + quantized
            # Update residual for next codebook
            residual = residual - quantized
            all_ids.append(ids)
            total_loss = total_loss + loss

        # (num_quantizers, batch_num, seq_len)
        all_ids = torch.stack(all_ids, dim=0)
        return quantized_sum, all_ids, total_loss


# Demonstrate RVQ
rvq = ResidualVQ(num_quantizers=4, codebook_size=1024, embed_dim=512)
rvq_quantized, rvq_ids, rvq_loss = rvq(continuous_tokens)

print(f"RVQ output shape: {rvq_quantized.shape}")
print(f"RVQ token IDs shape: {rvq_ids.shape}")
print(f"  → {rvq_ids.shape[0]} codebooks × {rvq_ids.shape[2]} time steps")
print(f"\nCodebook assignments for first 10 frames:")
for q_idx in range(4):
    print(f"  Codebook {q_idx}: {rvq_ids[q_idx, 0, :10].tolist()}")

# Measure reconstruction quality at each level
residual = continuous_tokens
cumulative = torch.zeros_like(continuous_tokens)
print(f"\nReconstruction error (MSE) at each RVQ level:")
for i, quantizer in enumerate(rvq.quantizers):
    q, _, _ = quantizer(residual)
    cumulative = cumulative + q
    residual = residual - q
    mse = F.mse_loss(cumulative, continuous_tokens).item()
    print(f"  After codebook {i}: MSE = {mse:.6f}")

---
# 3) Whisper Architecture Walkthrough

## 3.1 Overview: Encoder-Decoder for ASR

**Intuition:** Whisper (Radford et al., 2022) is a sequence-to-sequence model that transcribes speech. It uses an **encoder-decoder** Transformer, unlike the encoder-only approach used in multimodal LLMs.

```
┌─────────────────────────────────────────────────────────┐
│                    WHISPER ARCHITECTURE                   │
├─────────────────────────────────────────────────────────┤
│                                                           │
│  Audio (30s, 16kHz) → Mel Spectrogram (80, 3000)         │
│       │                                                   │
│       ▼                                                   │
│  Conv1D(80→d, k=3) + GELU + Conv1D(d→d, k=3, s=2)       │
│       │   ← Subsamples time by 2× (3000 → 1500)         │
│       ▼                                                   │
│  + Sinusoidal Positional Encoding                         │
│       │                                                   │
│       ▼                                                   │
│  Transformer Encoder (N layers of self-attention)         │
│       │                                                   │
│       ▼  Encoder output: (1, 1500, d)                    │
│       │                                                   │
│  Transformer Decoder                                      │
│    - Causal self-attention on text tokens                 │
│    - Cross-attention to encoder output                    │
│       │                                                   │
│       ▼                                                   │
│  Token logits → text                                      │
│                                                           │
└─────────────────────────────────────────────────────────┘
```

**Key design decisions:**
- **30-second chunks**: Whisper processes audio in fixed 30s windows. Shorter audio is zero-padded.
- **Conv subsampling**: Two 1D convolutions reduce the time dimension by 2×, turning 3000 spectrogram frames into 1500 encoder positions. This keeps self-attention tractable.
- **Multitask training**: Whisper is trained on multiple tasks (transcription, translation, language detection, timestamp prediction) using special tokens.

**Why encoder-decoder, not encoder-only?**
For ASR as a *standalone* task, the decoder provides the autoregressive text generation. In multimodal LLMs (Phi-4, MiniCPM-o), the *LLM itself* acts as the decoder, so they only need Whisper's *encoder* (or something like it).

In [ ]:
# ============================================================
# Whisper-style convolutional front-end from scratch
# This subsamples the mel spectrogram before the transformer
# ============================================================


class WhisperConvStem(nn.Module):
    """Whisper's convolutional front-end for mel spectrogram processing.

    Two 1D convolutions: the first with stride 1 for feature extraction,
    the second with stride 2 for temporal downsampling. GELU activation
    between them. This reduces the 3000 spectrogram frames to 1500
    encoder positions, making self-attention 4× cheaper.

    Design rationale: conv layers are better than linear projection at
    capturing local spectral patterns (formants, harmonics) before the
    transformer captures long-range dependencies.
    """

    def __init__(self, num_mel_bins: int = 80, model_dim: int = 512):
        super().__init__()
        # First conv: extract local features, no downsampling
        # (batch_num, num_mel_bins, time_steps) → (batch_num, model_dim, time_steps)
        self.conv1 = nn.Conv1d(
            in_channels=num_mel_bins,
            out_channels=model_dim,
            kernel_size=3,
            padding=1,
        )

        # Second conv: downsample time by 2× with stride=2
        # (batch_num, model_dim, time_steps) → (batch_num, model_dim, time_steps//2)
        self.conv2 = nn.Conv1d(
            in_channels=model_dim,
            out_channels=model_dim,
            kernel_size=3,
            stride=2,
            padding=1,
        )
        self.gelu = nn.GELU()

    def forward(self, mel_spec: torch.Tensor) -> torch.Tensor:
        """
        Args:
            mel_spec: (batch_num, num_mel_bins, time_steps)
        Returns:
            Features: (batch_num, time_steps//2, model_dim)
        """
        # (batch_num, num_mel_bins, time_steps) → (batch_num, model_dim, time_steps)
        x = self.gelu(self.conv1(mel_spec))

        # (batch_num, model_dim, time_steps) → (batch_num, model_dim, time_steps//2)
        x = self.gelu(self.conv2(x))

        # Transpose to (batch_num, time_steps//2, model_dim) for transformer
        return x.transpose(1, 2)


# Test the conv stem
conv_stem = WhisperConvStem(num_mel_bins=80, model_dim=512)
mel_batch = log_mel_spec.unsqueeze(0)  # (1, 80, num_frames)
conv_output = conv_stem(mel_batch)
print(f"Conv stem: {mel_batch.shape} → {conv_output.shape}")
print(f"  Time reduction: {mel_batch.shape[2]} → {conv_output.shape[1]} (2× downsample)")

## 3.2 Sinusoidal Positional Encoding

**Intuition:** Whisper uses *fixed* sinusoidal positional encodings (not learned). Each position gets a unique pattern of sine/cosine values at different frequencies. Low-frequency components encode coarse position; high-frequency ones encode fine position.

$$PE_{(pos, 2i)} = \sin\left(\frac{pos}{10000^{2i/d}}\right), \quad PE_{(pos, 2i+1)} = \cos\left(\frac{pos}{10000^{2i/d}}\right)$$

**Why fixed, not learned?** For audio, the max sequence length is always 1500 (30s at 20ms per frame after subsampling). Fixed sinusoidal encodings generalize to positions not seen during training, though this matters less when the length is fixed.

In [ ]:
# ============================================================
# Sinusoidal positional encoding from scratch
# ============================================================


class SinusoidalPositionalEncoding(nn.Module):
    """Fixed sinusoidal positional encoding as in Vaswani et al. (2017).

    Creates a (max_len, model_dim) matrix of position-dependent sinusoidal
    patterns. Even dimensions get sin, odd dimensions get cos, with
    frequencies geometrically spaced from 1/1 to 1/10000.
    """

    def __init__(self, model_dim: int, max_len: int = 1500):
        super().__init__()

        # Precompute the full positional encoding table
        # (max_len, model_dim)
        pe = torch.zeros(max_len, model_dim)
        position = torch.arange(0, max_len, dtype=torch.float32).unsqueeze(1)
        div_term = torch.exp(
            torch.arange(0, model_dim, 2, dtype=torch.float32)
            * (-math.log(10000.0) / model_dim)
        )

        # Even indices: sin, odd indices: cos
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)

        # Register as buffer (not a parameter, but moves with .to(device))
        self.register_buffer("pe", pe.unsqueeze(0))  # (1, max_len, model_dim)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x: (batch_num, seq_len, model_dim)
        Returns:
            x + positional encoding: (batch_num, seq_len, model_dim)
        """
        # (batch_num, seq_len, model_dim) + (1, seq_len, model_dim)
        return x + self.pe[:, : x.shape[1], :]


# Visualize the positional encoding patterns
pos_enc = SinusoidalPositionalEncoding(model_dim=512, max_len=1500)
pe_matrix = pos_enc.pe.squeeze(0)  # (1500, 512)

plt.figure(figsize=(12, 4))
plt.imshow(pe_matrix[:200, :64].numpy(), aspect="auto", cmap="RdBu_r")
plt.colorbar()
plt.xlabel("Embedding Dimension (first 64)")
plt.ylabel("Position")
plt.title("Sinusoidal Positional Encoding (first 200 positions × 64 dims)")
plt.tight_layout()
plt.show()

---
# 4) Build a Mini Audio Transformer Encoder From Scratch

## 4.1 Architecture Overview

**Intuition:** We build a complete Whisper-style audio encoder: conv stem → positional encoding → transformer layers. This encoder converts a mel spectrogram into a sequence of contextual embeddings that capture acoustic and linguistic information.

**Sample Input → Output:**
```
Input:  Mel spectrogram (batch_num, 80, time_steps)
Output: Audio embeddings (batch_num, time_steps//2, model_dim)
```

Each output vector is a *contextualized* representation of its corresponding ~20ms audio segment, enriched by attention over the full utterance.

**Why build from scratch?** To demystify the encoder and show it's just the same transformer architecture from NLP, applied to spectral features instead of word embeddings.

In [ ]:
# ============================================================
# Multi-head self-attention from scratch
# ============================================================


class MultiHeadSelfAttention(nn.Module):
    """Standard multi-head self-attention.

    Splits the model dimension into num_heads parallel attention heads,
    each computing scaled dot-product attention independently, then
    concatenates and projects back.

    For audio: each position (time frame) attends to all other positions,
    allowing the model to capture long-range acoustic dependencies
    (e.g., coarticulation effects spanning hundreds of milliseconds).
    """

    def __init__(self, model_dim: int, num_heads: int, dropout: float = 0.0):
        super().__init__()
        assert model_dim % num_heads == 0, "model_dim must be divisible by num_heads"
        self.num_heads = num_heads
        self.head_dim = model_dim // num_heads
        self.scale = self.head_dim ** -0.5

        # Combined QKV projection for efficiency
        # (batch_num, seq_len, model_dim) → (batch_num, seq_len, 3 * model_dim)
        self.qkv_proj = nn.Linear(model_dim, 3 * model_dim)

        # Output projection
        # (batch_num, seq_len, model_dim) → (batch_num, seq_len, model_dim)
        self.out_proj = nn.Linear(model_dim, model_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x: (batch_num, seq_len, model_dim)
        Returns:
            (batch_num, seq_len, model_dim)
        """
        batch_num, seq_len, model_dim = x.shape

        # Compute Q, K, V in one matmul
        # (batch_num, seq_len, model_dim) → (batch_num, seq_len, 3 * model_dim)
        qkv = self.qkv_proj(x)

        # Split into Q, K, V and reshape for multi-head attention
        # (batch_num, seq_len, 3*model_dim) → 3 × (batch_num, num_heads, seq_len, head_dim)
        qkv = qkv.reshape(batch_num, seq_len, 3, self.num_heads, self.head_dim)
        qkv = qkv.permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]

        # Scaled dot-product attention
        # (batch_num, num_heads, seq_len, head_dim) @ (batch_num, num_heads, head_dim, seq_len)
        # → (batch_num, num_heads, seq_len, seq_len)
        attn_weights = (q @ k.transpose(-2, -1)) * self.scale
        attn_weights = F.softmax(attn_weights, dim=-1)
        attn_weights = self.dropout(attn_weights)

        # Apply attention to values
        # (batch_num, num_heads, seq_len, seq_len) @ (batch_num, num_heads, seq_len, head_dim)
        # → (batch_num, num_heads, seq_len, head_dim)
        attn_output = attn_weights @ v

        # Concatenate heads and project
        # (batch_num, num_heads, seq_len, head_dim) → (batch_num, seq_len, model_dim)
        attn_output = attn_output.transpose(1, 2).reshape(batch_num, seq_len, model_dim)

        # (batch_num, seq_len, model_dim) → (batch_num, seq_len, model_dim)
        return self.out_proj(attn_output)

In [ ]:
# ============================================================
# Transformer encoder block and full audio encoder
# ============================================================


class TransformerEncoderBlock(nn.Module):
    """Pre-norm transformer encoder block: LN → MHSA → residual → LN → FFN → residual.

    Pre-norm (applying LayerNorm *before* each sub-layer) is used by Whisper
    and most modern transformers because it stabilizes training and removes
    the need for careful learning rate warmup.
    """

    def __init__(
        self,
        model_dim: int,
        num_heads: int,
        ffn_dim: int,
        dropout: float = 0.0,
    ):
        super().__init__()
        self.ln1 = nn.LayerNorm(model_dim)
        self.attn = MultiHeadSelfAttention(model_dim, num_heads, dropout)
        self.ln2 = nn.LayerNorm(model_dim)

        # Feed-forward network: expand → GELU → contract
        self.ffn = nn.Sequential(
            # (batch_num, seq_len, model_dim) → (batch_num, seq_len, ffn_dim)
            nn.Linear(model_dim, ffn_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            # (batch_num, seq_len, ffn_dim) → (batch_num, seq_len, model_dim)
            nn.Linear(ffn_dim, model_dim),
            nn.Dropout(dropout),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x: (batch_num, seq_len, model_dim)
        Returns:
            (batch_num, seq_len, model_dim)
        """
        # Self-attention with pre-norm and residual connection
        x = x + self.attn(self.ln1(x))

        # FFN with pre-norm and residual connection
        x = x + self.ffn(self.ln2(x))

        return x


class AudioTransformerEncoder(nn.Module):
    """Complete Whisper-style audio encoder.

    Architecture: Conv stem (2× downsample) → sinusoidal PE → N transformer blocks → LN.
    Converts a mel spectrogram into contextual audio embeddings.
    """

    def __init__(
        self,
        num_mel_bins: int = 80,
        model_dim: int = 512,
        num_heads: int = 8,
        num_layers: int = 6,
        ffn_dim: int = 2048,
        max_len: int = 1500,
        dropout: float = 0.0,
    ):
        super().__init__()
        self.conv_stem = WhisperConvStem(num_mel_bins, model_dim)
        self.pos_enc = SinusoidalPositionalEncoding(model_dim, max_len)

        self.layers = nn.ModuleList(
            [
                TransformerEncoderBlock(model_dim, num_heads, ffn_dim, dropout)
                for _ in range(num_layers)
            ]
        )
        self.final_ln = nn.LayerNorm(model_dim)

    def forward(self, mel_spec: torch.Tensor) -> torch.Tensor:
        """
        Args:
            mel_spec: (batch_num, num_mel_bins, time_steps)
        Returns:
            Audio embeddings: (batch_num, time_steps//2, model_dim)
        """
        # Conv subsampling: (batch_num, 80, T) → (batch_num, T//2, model_dim)
        x = self.conv_stem(mel_spec)

        # Add positional encoding
        # (batch_num, T//2, model_dim) → (batch_num, T//2, model_dim)
        x = self.pos_enc(x)

        # Pass through transformer layers
        for layer in self.layers:
            # (batch_num, T//2, model_dim) → (batch_num, T//2, model_dim)
            x = layer(x)

        # Final layer norm
        # (batch_num, T//2, model_dim) → (batch_num, T//2, model_dim)
        return self.final_ln(x)

In [ ]:
# ============================================================
# Instantiate and test the full audio encoder
# ============================================================

@dataclass
class AudioEncoderConfig:
    """Configuration for the audio transformer encoder."""
    num_mel_bins: int = 80
    model_dim: int = 512
    num_heads: int = 8
    num_layers: int = 6
    ffn_dim: int = 2048
    max_len: int = 1500
    dropout: float = 0.0


config = AudioEncoderConfig()
audio_encoder = AudioTransformerEncoder(
    num_mel_bins=config.num_mel_bins,
    model_dim=config.model_dim,
    num_heads=config.num_heads,
    num_layers=config.num_layers,
    ffn_dim=config.ffn_dim,
    max_len=config.max_len,
    dropout=config.dropout,
)

total_params = sum(p.numel() for p in audio_encoder.parameters())
print(f"Audio Encoder Parameters: {total_params:,} ({total_params / 1e6:.1f}M)")

# Test with our synthetic mel spectrogram
mel_input = log_mel_spec.unsqueeze(0)  # (1, 80, num_frames)
print(f"\nInput mel spectrogram: {mel_input.shape}")

with torch.no_grad():
    audio_embeddings = audio_encoder(mel_input)
print(f"Output audio embeddings: {audio_embeddings.shape}")
print(f"  Each embedding is a {config.model_dim}-dim vector representing ~20ms of audio")

# Also test with Whisper's expected 30-second input
whisper_mel = torch.randn(1, 80, 3000)  # 30s at 10ms per frame
with torch.no_grad():
    whisper_out = audio_encoder(whisper_mel)
print(f"\nWhisper-size input: {whisper_mel.shape} → output: {whisper_out.shape}")
print(f"  3000 spectrogram frames → 1500 encoder positions (2× downsampled)")

---
# 5) Connect Audio Encoder to LLM via MLP Projector

## 5.1 The Projector: Bridging Modalities

**Intuition:** The audio encoder produces embeddings of dimension `model_dim` (e.g., 512), but the LLM expects inputs of dimension `llm_dim` (e.g., 4096). The **MLP projector** maps between these spaces — the same architectural pattern used for vision in LLaVA, Phi-4, etc.

```
Audio encoder output    Projector              LLM embedding space
(1, 1500, 512)    →    MLP(512→4096)    →    (1, 1500, 4096)
```

**Why an MLP and not a linear layer?** The non-linear MLP can learn a more complex mapping between the audio and language embedding spaces. Empirically, a 2-layer MLP with GELU works better than a simple linear projection (as shown in LLaVA-1.5 and confirmed in Phi-4 Multimodal).

**The key insight:** This projector is the *only* new component needed to add audio to an existing LLM. The audio encoder is pretrained (like Whisper), the LLM is pretrained (like Llama), and only the projector is trained from scratch to align their representations.

**Sample Input → Output:**
```
Audio embeddings:   (batch_num, 1500, 512)  — from audio encoder
Projected:          (batch_num, 1500, 4096) — ready for LLM
Text embeddings:    (batch_num, seq_len, 4096) — from LLM tokenizer
Combined:           (batch_num, 1500 + seq_len, 4096) — full LLM input
```

In [ ]:
# ============================================================
# MLP Projector — bridges audio encoder to LLM embedding space
# ============================================================


class ModalityProjector(nn.Module):
    """2-layer MLP projector that maps encoder outputs to LLM embedding space.

    This is architecturally identical to the vision projector in LLaVA-1.5,
    Phi-4 Multimodal, and MiniCPM-o. The same projector design works for any
    modality because it's simply a learned linear transformation between two
    vector spaces, with a nonlinearity for expressiveness.

    Training strategy (Phi-4 Multimodal):
      Phase 1: Train only projector, freeze encoder + LLM
      Phase 2: Fine-tune projector + encoder, keep LLM frozen
      Phase 3: Full fine-tuning of all components
    """

    def __init__(self, encoder_dim: int, llm_dim: int):
        super().__init__()
        self.mlp = nn.Sequential(
            # (batch_num, seq_len, encoder_dim) → (batch_num, seq_len, llm_dim)
            nn.Linear(encoder_dim, llm_dim),
            nn.GELU(),
            # (batch_num, seq_len, llm_dim) → (batch_num, seq_len, llm_dim)
            nn.Linear(llm_dim, llm_dim),
        )

    def forward(self, encoder_output: torch.Tensor) -> torch.Tensor:
        """
        Args:
            encoder_output: (batch_num, seq_len, encoder_dim)
        Returns:
            Projected embeddings: (batch_num, seq_len, llm_dim)
        """
        return self.mlp(encoder_output)


# Instantiate projector
ENCODER_DIM = 512
LLM_DIM = 4096

audio_projector = ModalityProjector(encoder_dim=ENCODER_DIM, llm_dim=LLM_DIM)

proj_params = sum(p.numel() for p in audio_projector.parameters())
print(f"Projector parameters: {proj_params:,} ({proj_params / 1e6:.1f}M)")
print(f"  (Tiny compared to encoder {total_params / 1e6:.1f}M or a typical LLM)")

# Project audio embeddings to LLM space
with torch.no_grad():
    projected_audio = audio_projector(audio_embeddings)
print(f"\nProjected shape: {audio_embeddings.shape} → {projected_audio.shape}")

In [ ]:
# ============================================================
# Demonstrate the full concatenation with text tokens
# ============================================================


class AudioLanguageModel(nn.Module):
    """Multimodal model that combines audio and text for LLM processing.

    Architecture:
        audio → audio_encoder → projector → audio_tokens (in LLM space)
        text  → embedding_table                → text_tokens (in LLM space)
        concatenate [audio_tokens, text_tokens] → transformer_LLM → output

    This follows the exact same pattern as vision-language models:
    the modality-specific encoder converts raw input to dense vectors,
    the projector maps them to the LLM's embedding space, and they're
    simply concatenated with text token embeddings.
    """

    def __init__(
        self,
        audio_encoder: AudioTransformerEncoder,
        projector: ModalityProjector,
        vocab_size: int = 32000,
        llm_dim: int = 4096,
        num_layers: int = 2,
        num_heads: int = 8,
    ):
        super().__init__()
        self.audio_encoder = audio_encoder
        self.projector = projector

        # Text embedding (simulating the LLM's input embedding layer)
        # (vocab_size,) → (llm_dim,)
        self.text_embedding = nn.Embedding(vocab_size, llm_dim)

        # Simplified LLM decoder (just 2 layers for demonstration)
        self.llm_layers = nn.ModuleList(
            [
                TransformerEncoderBlock(llm_dim, num_heads, llm_dim * 4)
                for _ in range(num_layers)
            ]
        )
        self.ln_final = nn.LayerNorm(llm_dim)

        # LM head: project back to vocabulary logits
        # (batch_num, seq_len, llm_dim) → (batch_num, seq_len, vocab_size)
        self.lm_head = nn.Linear(llm_dim, vocab_size, bias=False)

    def forward(
        self,
        mel_spec: torch.Tensor,
        text_ids: torch.Tensor,
    ) -> torch.Tensor:
        """
        Args:
            mel_spec: (batch_num, num_mel_bins, time_steps)
            text_ids: (batch_num, text_seq_len)
        Returns:
            Logits: (batch_num, audio_seq_len + text_seq_len, vocab_size)
        """
        # Encode audio: mel → encoder → projector → LLM space
        # (batch_num, 80, T) → (batch_num, T//2, 512) → (batch_num, T//2, 4096)
        audio_features = self.audio_encoder(mel_spec)
        audio_tokens = self.projector(audio_features)

        # Embed text tokens
        # (batch_num, text_seq_len) → (batch_num, text_seq_len, llm_dim)
        text_tokens = self.text_embedding(text_ids)

        # Concatenate: [audio_tokens | text_tokens] along sequence dimension
        # (batch_num, T//2 + text_seq_len, llm_dim)
        combined = torch.cat([audio_tokens, text_tokens], dim=1)

        # Pass through LLM layers
        x = combined
        for layer in self.llm_layers:
            # (batch_num, total_seq_len, llm_dim) → (batch_num, total_seq_len, llm_dim)
            x = layer(x)
        x = self.ln_final(x)

        # Project to vocabulary
        # (batch_num, total_seq_len, llm_dim) → (batch_num, total_seq_len, vocab_size)
        return self.lm_head(x)


# Build the complete model
audio_lm = AudioLanguageModel(
    audio_encoder=audio_encoder,
    projector=audio_projector,
    vocab_size=32000,
    llm_dim=LLM_DIM,
    num_layers=2,
    num_heads=8,
)

total_alm_params = sum(p.numel() for p in audio_lm.parameters())
print(f"Full AudioLanguageModel: {total_alm_params:,} ({total_alm_params / 1e6:.1f}M)")
print(f"  Audio encoder: {total_params / 1e6:.1f}M")
print(f"  Projector: {proj_params / 1e6:.1f}M")
print(f"  LLM (2 layers, simplified): {(total_alm_params - total_params - proj_params) / 1e6:.1f}M")

In [ ]:
# ============================================================
# Test the full forward pass
# ============================================================

# Simulate a prompt: "Transcribe the following audio:"
text_prompt = torch.randint(0, 32000, (1, 10))  # 10 text tokens
mel_input = log_mel_spec.unsqueeze(0)  # (1, 80, num_frames)

print(f"Audio input:  {mel_input.shape} (mel spectrogram)")
print(f"Text input:   {text_prompt.shape} (token IDs)")

with torch.no_grad():
    logits = audio_lm(mel_input, text_prompt)

num_audio_tokens = mel_input.shape[2] // 2  # after conv downsampling
num_text_tokens = text_prompt.shape[1]

print(f"\nOutput logits: {logits.shape}")
print(f"  = {num_audio_tokens} audio tokens + {num_text_tokens} text tokens = {num_audio_tokens + num_text_tokens} total")
print(f"  Each position has logits over {logits.shape[-1]} vocabulary entries")
print(f"\n✓ Full pipeline: waveform → mel → encoder → projector → concat with text → LLM → logits")

---
# 6) Demo: Speech Understanding Through the LLM

## 6.1 Training Pipeline

**Intuition:** To train a model that understands speech via an LLM, we need (audio, text) pairs — the text is the target transcription. The training objective is the standard causal language modeling loss: predict the next text token given all previous audio + text tokens.

```
Input:   [<audio_tokens>, "Transcribe:", "The", "cat"]
Target:  [  —ignored—  , "Transcribe:", "The",  "cat", "sat"]
Loss:    only on text token positions (audio tokens are conditioning)
```

**Training phases (following Phi-4 Multimodal):**
1. **Projector warmup**: Freeze encoder + LLM, train only the MLP projector
2. **Joint training**: Unfreeze encoder, keep LLM frozen, train encoder + projector
3. **Full fine-tuning**: Optionally unfreeze everything (or use LoRA on the LLM)

This phased approach prevents the randomly-initialized projector from corrupting the pretrained encoder and LLM weights early in training.

In [ ]:
# ============================================================
# Demonstrate the training loop with synthetic data
# ============================================================


def create_synthetic_asr_batch(
    batch_num: int = 4,
    num_mel_bins: int = 80,
    time_steps: int = 300,
    text_seq_len: int = 20,
    vocab_size: int = 32000,
) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    """Generate a synthetic ASR training batch.

    Returns:
        mel_specs: (batch_num, num_mel_bins, time_steps)
        text_inputs: (batch_num, text_seq_len) — prompt + partial transcript
        text_targets: (batch_num, text_seq_len) — shifted by 1 for next-token prediction
    """
    mel_specs = torch.randn(batch_num, num_mel_bins, time_steps)
    text_inputs = torch.randint(1, vocab_size, (batch_num, text_seq_len))
    # Target is input shifted by 1 (standard causal LM setup)
    text_targets = torch.cat(
        [text_inputs[:, 1:], torch.randint(1, vocab_size, (batch_num, 1))], dim=1
    )
    return mel_specs, text_inputs, text_targets


def train_step(
    model: AudioLanguageModel,
    mel_specs: torch.Tensor,
    text_inputs: torch.Tensor,
    text_targets: torch.Tensor,
    optimizer: torch.optim.Optimizer,
) -> float:
    """Single training step with loss only on text positions."""
    model.train()
    optimizer.zero_grad()

    # Forward pass
    # (batch_num, audio_len + text_len, vocab_size)
    logits = model(mel_specs, text_inputs)

    # Extract logits for text positions only (skip audio positions)
    num_audio_tokens = mel_specs.shape[2] // 2
    text_logits = logits[:, num_audio_tokens:, :]

    # Cross-entropy loss on text predictions
    # (batch_num * text_seq_len, vocab_size) vs (batch_num * text_seq_len,)
    loss = F.cross_entropy(
        text_logits.reshape(-1, text_logits.shape[-1]),
        text_targets.reshape(-1),
    )

    loss.backward()
    optimizer.step()
    return loss.item()


# Training demonstration (Phase 1: projector-only warmup)
print("Phase 1: Projector warmup (freeze encoder + LLM)")
print("=" * 50)

# Freeze audio encoder
for param in audio_lm.audio_encoder.parameters():
    param.requires_grad = False

# Freeze LLM layers
for param in audio_lm.llm_layers.parameters():
    param.requires_grad = False
for param in audio_lm.text_embedding.parameters():
    param.requires_grad = False
for param in audio_lm.lm_head.parameters():
    param.requires_grad = False

trainable = sum(p.numel() for p in audio_lm.parameters() if p.requires_grad)
total = sum(p.numel() for p in audio_lm.parameters())
print(f"Trainable: {trainable:,} / {total:,} ({100*trainable/total:.1f}%)")

optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, audio_lm.parameters()),
    lr=1e-3,
)

# Run a few training steps
losses = []
for step in range(20):
    mel_specs, text_inputs, text_targets = create_synthetic_asr_batch(batch_num=4)
    loss = train_step(audio_lm, mel_specs, text_inputs, text_targets, optimizer)
    losses.append(loss)
    if step % 5 == 0:
        print(f"  Step {step:3d}: loss = {loss:.4f}")

plt.figure(figsize=(8, 3))
plt.plot(losses)
plt.xlabel("Step")
plt.ylabel("Loss")
plt.title("Phase 1: Projector Warmup Training Loss")
plt.tight_layout()
plt.show()

print("\n✓ In practice, Phase 1 runs for ~1 epoch on ASR data")
print("  Then Phase 2 unfreezes the encoder for joint training")
print("  Then Phase 3 optionally fine-tunes the full model")

## 6.2 Inference: Greedy Decoding

**Intuition:** At inference time, we feed the audio and a prompt (e.g., "Transcribe:") to the model, then autoregressively generate text tokens one at a time. The audio tokens provide the acoustic context; each new text token is conditioned on all audio + previously generated text.

```
Step 0: [audio_tokens, "Transcribe:"] → predict "The"
Step 1: [audio_tokens, "Transcribe:", "The"] → predict "quick"
Step 2: [audio_tokens, "Transcribe:", "The", "quick"] → predict "brown"
...
```

In [ ]:
# ============================================================
# Greedy decoding for speech-to-text generation
# ============================================================


@torch.no_grad()
def greedy_decode(
    model: AudioLanguageModel,
    mel_spec: torch.Tensor,
    prompt_ids: torch.Tensor,
    max_new_tokens: int = 50,
    eos_token_id: int = 2,
) -> torch.Tensor:
    """Autoregressively generate text from audio + text prompt.

    At each step, we run the full model on [audio_tokens, text_so_far],
    take the argmax of the last position's logits, and append it.

    Args:
        model: AudioLanguageModel instance.
        mel_spec: (1, num_mel_bins, time_steps)
        prompt_ids: (1, prompt_len) — text prompt token IDs
        max_new_tokens: Maximum number of tokens to generate.
        eos_token_id: Stop when this token is predicted.

    Returns:
        Generated token IDs: (1, prompt_len + num_generated)
    """
    model.eval()
    current_ids = prompt_ids.clone()

    for _ in range(max_new_tokens):
        # Full forward pass with current text sequence
        # (1, audio_len + text_len, vocab_size)
        logits = model(mel_spec, current_ids)

        # Take logits at the last position (next-token prediction)
        # (1, vocab_size)
        next_logits = logits[:, -1, :]

        # Greedy: pick the highest-probability token
        # (1,)
        next_token = next_logits.argmax(dim=-1, keepdim=True)

        # Append to sequence
        # (1, text_len) → (1, text_len + 1)
        current_ids = torch.cat([current_ids, next_token], dim=1)

        # Stop at EOS
        if next_token.item() == eos_token_id:
            break

    return current_ids


# Demo: generate tokens from audio (model is untrained, so output is random)
prompt = torch.tensor([[1, 100, 200, 300]])  # Fake "Transcribe:" prompt
generated = greedy_decode(audio_lm, mel_input, prompt, max_new_tokens=15)
print(f"Input prompt: {prompt.shape[1]} tokens")
print(f"Generated:    {generated.shape[1]} tokens (including prompt)")
print(f"Token IDs: {generated[0].tolist()}")
print(f"\n(Output is random — model is untrained. With real training on ASR data,")
print(f" these would be token IDs encoding the spoken words.)")

---
# 7) Unified Audio-Vision-Text: Phi-4 Multimodal & MiniCPM-o

## 7.1 The Omni-Modal Pattern

**Key Insight:** Once you have the `encoder → projector → LLM` pattern working for one modality, adding more modalities is just adding more encoder+projector pairs. The LLM doesn't care where the embeddings came from — it sees a flat sequence of vectors.

```
┌───────────────────────────────────────────────────────────────────┐
│                    OMNI-MODAL LLM ARCHITECTURE                    │
│                  (Phi-4 Multimodal / MiniCPM-o)                   │
├───────────────────────────────────────────────────────────────────┤
│                                                                   │
│  Image → ViT Encoder → Vision Projector ──┐                      │
│                                            │                      │
│  Audio → Whisper Encoder → Audio Projector ─┤→ Concat → LLM      │
│                                            │                      │
│  Text  → Token Embedding ─────────────────┘                      │
│                                                                   │
└───────────────────────────────────────────────────────────────────┘
```

## 7.2 Phi-4 Multimodal (Microsoft, 2025)

**Architecture specifics:**
- Base LLM: Phi-4 (14B parameters)
- Vision: Custom ViT encoder with dynamic resolution (any aspect ratio)
- Audio: Modified Whisper encoder for speech understanding
- Projectors: Separate 2-layer MLPs for each modality
- Training: 3-phase curriculum (projector warmup → encoder+projector → full)

**What's novel:**
- Uses **mixture of LoRAs** — different LoRA adapters for different modalities, dynamically selected based on which modality is active
- Supports interleaved multi-modal inputs (audio + image + text in arbitrary order)

## 7.3 MiniCPM-o (OpenBMB, 2025)

**Architecture specifics:**
- Base LLM: MiniCPM-3 (4B parameters — much smaller than Phi-4)
- Vision: SigLIP-based encoder with dynamic slicing
- Audio: Whisper-large-v3 encoder, but with a twist — uses **audio compressor** (cross-attention pooling) to reduce the number of audio tokens before projection
- End-to-end speech generation: can both understand AND produce speech

**What's novel:**
- **Audio compressor**: Instead of projecting all 1500 encoder positions, uses a set of learnable queries (e.g., 64) that cross-attend to the encoder output, reducing 1500 positions to 64. This is the same idea as Perceiver/Q-Former.
- **Streaming speech**: Can produce audio output token-by-token for real-time conversation

Let's implement the key components that differentiate these architectures.

In [ ]:
# ============================================================
# Audio Compressor (Q-Former style cross-attention pooling)
# Used by MiniCPM-o to reduce audio token count before the LLM
# ============================================================


class CrossAttentionPooling(nn.Module):
    """Reduce a variable-length encoder sequence to a fixed number of tokens.

    Uses learnable query vectors that cross-attend to the encoder output.
    This is the same principle as Perceiver (Jaegle et al., 2021) and
    Q-Former (Li et al., 2023) — a small set of queries "reads" the
    relevant information from a long input.

    Motivation: 1500 audio tokens is expensive for LLM attention (O(n²)).
    Compressing to 64 tokens reduces audio's cost by ~500× while retaining
    the information needed for speech understanding.
    """

    def __init__(
        self,
        model_dim: int = 512,
        num_queries: int = 64,
        num_heads: int = 8,
    ):
        super().__init__()
        self.num_queries = num_queries
        self.head_dim = model_dim // num_heads
        self.num_heads = num_heads
        self.scale = self.head_dim ** -0.5

        # Learnable query vectors — these are the "compressed" representations
        # (num_queries, model_dim)
        self.queries = nn.Parameter(torch.randn(num_queries, model_dim) * 0.02)

        # Cross-attention projections
        # Queries come from learnable vectors, Keys/Values come from encoder
        self.q_proj = nn.Linear(model_dim, model_dim)
        self.k_proj = nn.Linear(model_dim, model_dim)
        self.v_proj = nn.Linear(model_dim, model_dim)
        self.out_proj = nn.Linear(model_dim, model_dim)
        self.ln = nn.LayerNorm(model_dim)

    def forward(self, encoder_output: torch.Tensor) -> torch.Tensor:
        """
        Args:
            encoder_output: (batch_num, seq_len, model_dim)  e.g., (1, 1500, 512)
        Returns:
            Compressed: (batch_num, num_queries, model_dim)  e.g., (1, 64, 512)
        """
        batch_num = encoder_output.shape[0]

        # Expand queries for the batch
        # (num_queries, model_dim) → (batch_num, num_queries, model_dim)
        q = self.queries.unsqueeze(0).expand(batch_num, -1, -1)

        # Project Q (from queries) and K, V (from encoder)
        # (batch_num, num_queries, model_dim) → (batch_num, num_heads, num_queries, head_dim)
        q = self.q_proj(q).reshape(batch_num, self.num_queries, self.num_heads, self.head_dim).transpose(1, 2)

        # (batch_num, seq_len, model_dim) → (batch_num, num_heads, seq_len, head_dim)
        k = self.k_proj(encoder_output).reshape(batch_num, -1, self.num_heads, self.head_dim).transpose(1, 2)
        v = self.v_proj(encoder_output).reshape(batch_num, -1, self.num_heads, self.head_dim).transpose(1, 2)

        # Cross-attention: queries attend to encoder output
        # (batch_num, num_heads, num_queries, head_dim) @ (batch_num, num_heads, head_dim, seq_len)
        # → (batch_num, num_heads, num_queries, seq_len)
        attn_weights = (q @ k.transpose(-2, -1)) * self.scale
        attn_weights = F.softmax(attn_weights, dim=-1)

        # (batch_num, num_heads, num_queries, seq_len) @ (batch_num, num_heads, seq_len, head_dim)
        # → (batch_num, num_heads, num_queries, head_dim)
        attn_out = attn_weights @ v

        # Concatenate heads and project
        # (batch_num, num_heads, num_queries, head_dim) → (batch_num, num_queries, model_dim)
        attn_out = attn_out.transpose(1, 2).reshape(batch_num, self.num_queries, -1)
        return self.ln(self.out_proj(attn_out))


# Demonstrate compression
compressor = CrossAttentionPooling(model_dim=512, num_queries=64, num_heads=8)

# Simulate encoder output for 30s of audio
encoder_out = torch.randn(1, 1500, 512)
with torch.no_grad():
    compressed = compressor(encoder_out)

print(f"Encoder output: {encoder_out.shape} (1500 tokens from 30s audio)")
print(f"Compressed:     {compressed.shape} (64 tokens — 23× reduction)")
print(f"\nThis reduces LLM computation on audio from O(1500²) to O(64²) = ~550× cheaper")

In [ ]:
# ============================================================
# Modality-Specific LoRA (Phi-4 Multimodal pattern)
# ============================================================


class LoRALinear(nn.Module):
    """Low-Rank Adaptation layer.

    Adds a trainable low-rank decomposition BA to a frozen pretrained
    weight W: output = Wx + BAx. Rank r << min(in, out) so the adaptation
    has far fewer parameters than full fine-tuning.

    In Phi-4 Multimodal, different LoRA weights are used for different
    modalities, allowing modality-specific adaptation of the frozen LLM.
    """

    def __init__(
        self,
        in_features: int,
        out_features: int,
        rank: int = 16,
        alpha: float = 32.0,
    ):
        super().__init__()
        # Frozen pretrained weight
        self.frozen_linear = nn.Linear(in_features, out_features)
        self.frozen_linear.weight.requires_grad = False
        if self.frozen_linear.bias is not None:
            self.frozen_linear.bias.requires_grad = False

        # Low-rank trainable matrices
        # (in_features,) → (rank,) → (out_features,)
        self.lora_A = nn.Linear(in_features, rank, bias=False)
        self.lora_B = nn.Linear(rank, out_features, bias=False)

        # Scaling factor
        self.scaling = alpha / rank

        # Initialize B to zero so LoRA starts as identity
        nn.init.zeros_(self.lora_B.weight)
        nn.init.kaiming_normal_(self.lora_A.weight)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x: (batch_num, seq_len, in_features)
        Returns:
            (batch_num, seq_len, out_features)
        """
        # Frozen path: Wx
        frozen_out = self.frozen_linear(x)

        # LoRA path: scaling * B(A(x))
        # (batch_num, seq_len, in_features) → (batch_num, seq_len, rank)
        # → (batch_num, seq_len, out_features)
        lora_out = self.lora_B(self.lora_A(x)) * self.scaling

        return frozen_out + lora_out


class MixtureOfLoRA(nn.Module):
    """Modality-specific LoRA routing (Phi-4 Multimodal pattern).

    Maintains separate LoRA adapters for each modality (vision, audio, text).
    Based on which modality tokens are present, the corresponding LoRA
    adapter is activated. This allows the LLM to specialize its processing
    for different input types without interference.
    """

    def __init__(
        self,
        in_features: int,
        out_features: int,
        rank: int = 16,
        modalities: Tuple[str, ...] = ("vision", "audio", "text"),
    ):
        super().__init__()
        # Shared frozen weight
        self.frozen_linear = nn.Linear(in_features, out_features)
        self.frozen_linear.weight.requires_grad = False

        # Per-modality LoRA adapters
        self.lora_adapters = nn.ModuleDict(
            {
                mod: nn.Sequential(
                    nn.Linear(in_features, rank, bias=False),
                    nn.Linear(rank, out_features, bias=False),
                )
                for mod in modalities
            }
        )

    def forward(self, x: torch.Tensor, modality: str) -> torch.Tensor:
        """
        Args:
            x: (batch_num, seq_len, in_features)
            modality: Which modality's LoRA to use
        Returns:
            (batch_num, seq_len, out_features)
        """
        frozen_out = self.frozen_linear(x)
        lora_out = self.lora_adapters[modality](x)
        return frozen_out + lora_out


# Demonstrate Mixture of LoRA
mol = MixtureOfLoRA(in_features=4096, out_features=4096, rank=16)

dummy_input = torch.randn(1, 10, 4096)
with torch.no_grad():
    vision_out = mol(dummy_input, "vision")
    audio_out = mol(dummy_input, "audio")
    text_out = mol(dummy_input, "text")

# Verify different modalities produce different outputs
print("Mixture of LoRA — same input, different modality adapters:")
print(f"  Vision LoRA output norm: {vision_out.norm().item():.2f}")
print(f"  Audio LoRA output norm:  {audio_out.norm().item():.2f}")
print(f"  Text LoRA output norm:   {text_out.norm().item():.2f}")
print(f"  Vision vs Audio diff:    {(vision_out - audio_out).norm().item():.2f}")
print(f"\n  → Different adapters produce different transformations of the same input")

mol_params = sum(p.numel() for p in mol.lora_adapters.parameters())
frozen_params = sum(p.numel() for p in mol.frozen_linear.parameters())
print(f"\nAll LoRA adapters: {mol_params:,} params ({mol_params/frozen_params*100:.1f}% of frozen)")

In [ ]:
# ============================================================
# Full omni-modal architecture demonstration
# ============================================================


class OmniModalLLM(nn.Module):
    """Simplified omni-modal architecture combining vision, audio, and text.

    Demonstrates the key architectural pattern used by Phi-4 Multimodal
    and MiniCPM-o: each modality has its own encoder and projector,
    all feeding into the same LLM.

    The LLM sees a flat sequence of embeddings and doesn't inherently
    know which modality each token came from — this information is only
    encoded in the embedding values themselves (via modality-specific
    encoders) and optionally via special separator tokens.
    """

    def __init__(
        self,
        llm_dim: int = 512,  # Small for demo
        vocab_size: int = 32000,
        num_mel_bins: int = 80,
        audio_encoder_dim: int = 256,
        vision_patch_dim: int = 768,
        num_audio_queries: int = 64,
    ):
        super().__init__()

        # --- Audio pathway ---
        self.audio_encoder = AudioTransformerEncoder(
            num_mel_bins=num_mel_bins,
            model_dim=audio_encoder_dim,
            num_heads=4,
            num_layers=2,
            ffn_dim=audio_encoder_dim * 4,
        )
        # Optional: compress audio tokens before projection
        self.audio_compressor = CrossAttentionPooling(
            model_dim=audio_encoder_dim,
            num_queries=num_audio_queries,
            num_heads=4,
        )
        self.audio_projector = ModalityProjector(audio_encoder_dim, llm_dim)

        # --- Vision pathway (simplified — just a projector) ---
        self.vision_projector = ModalityProjector(vision_patch_dim, llm_dim)

        # --- Text pathway ---
        self.text_embedding = nn.Embedding(vocab_size, llm_dim)

        # --- Shared LLM backbone ---
        self.llm = nn.Sequential(
            TransformerEncoderBlock(llm_dim, num_heads=8, ffn_dim=llm_dim * 4),
            TransformerEncoderBlock(llm_dim, num_heads=8, ffn_dim=llm_dim * 4),
            nn.LayerNorm(llm_dim),
        )
        self.lm_head = nn.Linear(llm_dim, vocab_size, bias=False)

    def forward(
        self,
        text_ids: torch.Tensor,
        mel_spec: Optional[torch.Tensor] = None,
        vision_features: Optional[torch.Tensor] = None,
    ) -> torch.Tensor:
        """
        Args:
            text_ids: (batch_num, text_seq_len)
            mel_spec: Optional (batch_num, num_mel_bins, time_steps)
            vision_features: Optional (batch_num, num_patches, vision_patch_dim)
        Returns:
            Logits: (batch_num, total_seq_len, vocab_size)
        """
        embeddings_list = []

        # Process audio if present
        if mel_spec is not None:
            # (batch_num, 80, T) → (batch_num, T//2, audio_dim)
            audio_enc = self.audio_encoder(mel_spec)
            # (batch_num, T//2, audio_dim) → (batch_num, 64, audio_dim)
            audio_compressed = self.audio_compressor(audio_enc)
            # (batch_num, 64, audio_dim) → (batch_num, 64, llm_dim)
            audio_tokens = self.audio_projector(audio_compressed)
            embeddings_list.append(audio_tokens)

        # Process vision if present
        if vision_features is not None:
            # (batch_num, num_patches, vision_dim) → (batch_num, num_patches, llm_dim)
            vision_tokens = self.vision_projector(vision_features)
            embeddings_list.append(vision_tokens)

        # Always process text
        # (batch_num, text_seq_len) → (batch_num, text_seq_len, llm_dim)
        text_tokens = self.text_embedding(text_ids)
        embeddings_list.append(text_tokens)

        # Concatenate all modalities along sequence dimension
        # (batch_num, total_seq_len, llm_dim)
        combined = torch.cat(embeddings_list, dim=1)

        # Pass through shared LLM
        # (batch_num, total_seq_len, llm_dim) → (batch_num, total_seq_len, llm_dim)
        hidden = self.llm(combined)

        # (batch_num, total_seq_len, llm_dim) → (batch_num, total_seq_len, vocab_size)
        return self.lm_head(hidden)


# Build and test the omni-modal model
omni_model = OmniModalLLM(llm_dim=512, audio_encoder_dim=256)

omni_params = sum(p.numel() for p in omni_model.parameters())
print(f"Omni-Modal LLM: {omni_params:,} ({omni_params / 1e6:.1f}M) params")

# Test with all three modalities
text_ids = torch.randint(0, 32000, (1, 15))
mel_input = torch.randn(1, 80, 300)
vision_input = torch.randn(1, 196, 768)  # 14×14 ViT patches

with torch.no_grad():
    # Text only
    out_text = omni_model(text_ids)
    print(f"\nText only:           {text_ids.shape} → {out_text.shape}")

    # Audio + text
    out_audio = omni_model(text_ids, mel_spec=mel_input)
    print(f"Audio + text:        audio {mel_input.shape} + text {text_ids.shape} → {out_audio.shape}")

    # Vision + text
    out_vision = omni_model(text_ids, vision_features=vision_input)
    print(f"Vision + text:       vision {vision_input.shape} + text {text_ids.shape} → {out_vision.shape}")

    # All three modalities
    out_all = omni_model(text_ids, mel_spec=mel_input, vision_features=vision_input)
    print(f"Audio + Vision + text: all three → {out_all.shape}")
    print(f"  = 64 audio + 196 vision + 15 text = {64 + 196 + 15} total tokens")

## 7.4 Architecture Comparison Table

| Component | Whisper (2022) | Phi-4 Multimodal (2025) | MiniCPM-o (2025) |
|-----------|---------------|------------------------|-------------------|
| **LLM** | N/A (decoder is task-specific) | Phi-4 (14B) | MiniCPM-3 (4B) |
| **Audio Encoder** | Custom Transformer | Modified Whisper | Whisper-large-v3 |
| **Audio Tokens** | 1500 (30s, no compression) | ~1500 | 64 (compressed via Q-Former) |
| **Vision Encoder** | N/A | Custom ViT | SigLIP |
| **Projector** | N/A (encoder-decoder) | 2-layer MLP per modality | MLP + compressor |
| **LLM Adaptation** | N/A | Mixture of LoRA | Full fine-tuning |
| **Speech Output** | Text only | Text only | Text + speech |
| **Training Data** | 680K hours audio | Billions of tokens mixed | Smaller, curated |

---
# 8) Bonus: Speech Synthesis Overview (TTS)

## 8.1 The Reverse Direction: Text → Speech

**Intuition:** TTS inverts the speech understanding pipeline. Instead of `audio → encoder → LLM → text`, we need `text → model → audio`. The core challenge is that the mapping is *one-to-many*: the same text can be spoken in countless ways (different voices, speeds, emotions, accents).

## 8.2 Major TTS Architectures

**Voicebox (Meta, 2023) — Flow Matching:**
```
Text + audio context → Transformer → continuous mel spectrogram (via flow matching)
                                    → vocoder → waveform
```
- Uses **conditional flow matching**: learns a vector field that transforms noise into mel spectrograms, conditioned on text and optional audio context
- Can do in-context learning: given a 3s audio sample, generates speech in that voice
- Non-autoregressive: generates all frames in parallel (fast)

**F5-TTS (2024) — Flow Matching with Diffusion Transformer:**
```
Text (character-level) + reference audio → DiT → mel spectrogram → vocoder → waveform
```
- Simplifies Voicebox by removing the duration model
- Uses text infilling: pads text to match target length, lets the model figure out alignment
- Diffusion Transformer (DiT) architecture — same as used in image generation (Stable Diffusion 3)

**ChatTTS (2024) — Autoregressive + Flow:**
```
Text → GPT-style LLM → discrete audio tokens → flow-based decoder → waveform
```
- Generates discrete audio codes autoregressively (like a language model generating text)
- Supports prosody control via special tokens (laughter, pauses, emphasis)
- Designed for conversational speech with natural interruptions

## 8.3 Connection to Understanding Models

**The unified vision (MiniCPM-o):** Use the same LLM for both understanding and generation:
```
Understanding: audio → encoder → projector → LLM → text tokens
Generation:    text → LLM → audio tokens → decoder → waveform
```

The LLM operates in a shared token space where both text and audio tokens coexist, enabling natural turn-taking in conversation.

In [ ]:
# ============================================================
# Simplified TTS decoder demonstration (flow matching concept)
# ============================================================


class SimplifiedFlowMatchingTTS(nn.Module):
    """Minimal flow matching decoder for TTS (conceptual demonstration).

    Flow matching learns a vector field v(x_t, t) that transports samples
    from a noise distribution (t=0) to the data distribution (t=1).
    The training objective is:
        L = E_t ||v_θ(x_t, t, cond) - (x_1 - x_0)||²

    where x_t = (1-t)·x_0 + t·x_1 is a linear interpolation between
    noise (x_0) and data (x_1), and cond is the text conditioning.

    At inference: start from noise, integrate v_θ over t ∈ [0, 1] using
    Euler steps to get a mel spectrogram.
    """

    def __init__(self, mel_dim: int = 80, text_dim: int = 512, hidden_dim: int = 256):
        super().__init__()
        # Predict velocity field conditioned on noisy mel + time + text
        self.net = nn.Sequential(
            # (mel_dim + 1 + text_dim,) → (hidden_dim,)
            nn.Linear(mel_dim + 1 + text_dim, hidden_dim),
            nn.GELU(),
            # (hidden_dim,) → (hidden_dim,)
            nn.Linear(hidden_dim, hidden_dim),
            nn.GELU(),
            # (hidden_dim,) → (mel_dim,)
            nn.Linear(hidden_dim, mel_dim),
        )

    def forward(
        self,
        x_t: torch.Tensor,
        t: torch.Tensor,
        text_cond: torch.Tensor,
    ) -> torch.Tensor:
        """
        Args:
            x_t: Noisy mel frames (batch_num, num_frames, mel_dim)
            t: Time step (batch_num, num_frames, 1)
            text_cond: Text conditioning (batch_num, num_frames, text_dim)
        Returns:
            Predicted velocity: (batch_num, num_frames, mel_dim)
        """
        # Concatenate inputs along feature dimension
        # (batch_num, num_frames, mel_dim + 1 + text_dim)
        inp = torch.cat([x_t, t, text_cond], dim=-1)
        return self.net(inp)

    def compute_loss(
        self,
        x_1: torch.Tensor,
        text_cond: torch.Tensor,
    ) -> torch.Tensor:
        """Flow matching training loss.

        Args:
            x_1: Target mel spectrogram (batch_num, num_frames, mel_dim)
            text_cond: Text conditioning (batch_num, num_frames, text_dim)
        Returns:
            MSE loss between predicted and true velocity
        """
        batch_num, num_frames, mel_dim = x_1.shape

        # Sample random noise (source distribution)
        x_0 = torch.randn_like(x_1)

        # Sample random time for each example
        t = torch.rand(batch_num, num_frames, 1, device=x_1.device)

        # Linear interpolation: x_t = (1-t)·x_0 + t·x_1
        x_t = (1 - t) * x_0 + t * x_1

        # True velocity: dx/dt = x_1 - x_0 (derivative of linear interpolation)
        true_velocity = x_1 - x_0

        # Predicted velocity
        pred_velocity = self.forward(x_t, t, text_cond)

        return F.mse_loss(pred_velocity, true_velocity)

    @torch.no_grad()
    def sample(
        self,
        text_cond: torch.Tensor,
        num_frames: int,
        num_steps: int = 20,
    ) -> torch.Tensor:
        """Generate mel spectrogram via Euler integration of the velocity field.

        Args:
            text_cond: (batch_num, num_frames, text_dim)
            num_frames: Number of mel spectrogram frames to generate
            num_steps: Number of Euler integration steps
        Returns:
            Generated mel: (batch_num, num_frames, mel_dim)
        """
        batch_num = text_cond.shape[0]
        mel_dim = 80
        dt = 1.0 / num_steps

        # Start from noise (t=0)
        x = torch.randn(batch_num, num_frames, mel_dim)

        # Euler integration from t=0 to t=1
        for step in range(num_steps):
            t_val = step / num_steps
            t = torch.full((batch_num, num_frames, 1), t_val)

            # Predict velocity and take Euler step
            velocity = self.forward(x, t, text_cond)
            x = x + velocity * dt

        return x


# Demonstrate the TTS flow
tts_model = SimplifiedFlowMatchingTTS(mel_dim=80, text_dim=512, hidden_dim=256)

# Simulate training
text_cond = torch.randn(2, 100, 512)  # Text conditioning for 100 frames
target_mel = torch.randn(2, 100, 80)  # Target mel spectrogram

loss = tts_model.compute_loss(target_mel, text_cond)
print(f"Flow matching training loss: {loss.item():.4f}")

# Simulate inference (generation)
generated_mel = tts_model.sample(text_cond[:1], num_frames=100, num_steps=20)
print(f"Generated mel spectrogram: {generated_mel.shape}")

fig, axes = plt.subplots(1, 2, figsize=(12, 3))
axes[0].imshow(target_mel[0].T.numpy(), aspect="auto", origin="lower", cmap="viridis")
axes[0].set_title("Target Mel Spectrogram")
axes[0].set_ylabel("Mel Bin")

axes[1].imshow(generated_mel[0].T.numpy(), aspect="auto", origin="lower", cmap="viridis")
axes[1].set_title("Generated Mel (untrained — random)")
axes[1].set_ylabel("Mel Bin")
axes[1].set_xlabel("Time Frame")

plt.tight_layout()
plt.show()
print("\n(Generated spectrogram is noise — model is untrained. With real training,")
print(" this would produce a mel spectrogram that a vocoder can convert to speech.)")

---
# Summary & Key Takeaways

## The Universal Multimodal Pattern

The central message of this chapter:

```
Any modality → Modality-specific encoder → Projector (MLP) → LLM
```

This pattern is *identical* for:
- **Vision**: Image → ViT → MLP → LLM (Ch 4-5)
- **Audio**: Waveform → Mel spectrogram → Audio Transformer → MLP → LLM (this chapter)
- **Video**: Frames → ViT per frame → temporal pooling → MLP → LLM
- **Any future modality**: Sensor data → domain encoder → MLP → LLM

## What We Built From Scratch

1. **Mel spectrogram pipeline**: STFT → mel filterbank → log compression
2. **Vector quantization**: VQ + Residual VQ for discrete audio tokens
3. **Whisper-style audio encoder**: Conv stem + sinusoidal PE + Transformer
4. **Modality projector**: 2-layer MLP bridging encoder and LLM spaces
5. **Audio-Language Model**: Full encoder → projector → LLM pipeline
6. **Cross-attention compressor**: Q-Former style token reduction (MiniCPM-o)
7. **Mixture of LoRA**: Modality-specific LLM adaptation (Phi-4 Multimodal)
8. **Flow matching TTS**: Velocity field for mel spectrogram generation

## Paper References

- Radford et al. (2022). *Robust Speech Recognition via Large-Scale Weak Supervision.* (Whisper)
- Abrantes et al. (2025). *Phi-4-Multimodal Technical Report.* (Phi-4 MM)
- Yao et al. (2025). *MiniCPM-o: A Fully Open-Source Multimodal Model.* (MiniCPM-o)
- Défossez et al. (2022). *High Fidelity Neural Audio Compression.* (EnCodec)
- Le et al. (2023). *Voicebox: Text-Guided Multilingual Universal Speech Generation at Scale.*
- Chen et al. (2024). *F5-TTS: A Fairytaler that Fakes Fluent and Faithful Speech.*